In [1]:
import numpy as np
import torch
import torch.nn as nn

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## Embedding Posicional

In [3]:
class EmbeddingPosicional(nn.Module):
    def __init__(self, d_embedding, vocab_size, seq_len):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(1, seq_len, d_embedding) * 0.02) # Inicialización pequeña
        self.embedding = nn.Embedding(vocab_size, d_embedding)
    
    def forward(self, x):
        out = self.embedding(x) # devuelve (b, n, d)
        _, n, _ = out.shape
        out = out + self.weights[:,:n,:]
        return out

## Capa Feedforward

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.l1 = nn.Linear(d_model, d_model*4)
        self.silu = nn.SiLU()
        self.l2 = nn.Linear(d_model*4, d_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm(x)
        x = self.l1(x)
        x = self.silu(x)
        x = self.l2(x)
        x = self.dropout(x)
        return x

## Capa de atención

In [5]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_head, d_head, use_mask=False):
        super().__init__()
        self.qw = nn.Linear(d_model, d_head*n_head)
        self.kw = nn.Linear(d_model, d_head*n_head)
        self.vw = nn.Linear(d_model, d_head*n_head)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)
        self.out = nn.Linear(d_head*n_head, d_model)

        self.n_head = n_head
        self.d_head = d_head
        self.scale = d_head ** 0.5
        self.use_mask = use_mask
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        b, n, d = x.shape
        x = self.norm(x)

        q = self.qw(x)
        k = self.kw(x)
        v = self.vw(x)

        q = q.reshape(b, -1, self.n_head, self.d_head)
        k = k.reshape(b, -1, self.n_head, self.d_head)
        v = v.reshape(b, -1, self.n_head, self.d_head)

        #print(q.shape)
        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale
        
        if self.use_mask:
            _, _, _, n = scores.shape
            mask = torch.tril(torch.ones(1, 1, n, n))   # (n, n)

            scores = scores.masked_fill(mask[:, :, :n, :n] == 0, float('-inf'))

        softmax_scores = scores.softmax(dim=-1)
        att = self.dropout(softmax_scores)
        out = torch.einsum('bhij,bjhd->bihd', att, v)
        #print(out.shape)
        out = self.dropout(out).reshape(b, -1, self.d_head*self.n_head)
        #print(out.shape)
        out = self.out(out)
        return out, att

## Encoder

In [6]:
class Encoder(nn.Module):
    def __init__(self, d_model=16, vocab_size=20, seq_len=32, nb_layers=5, n_head=2, d_head=8):
        super().__init__()
        self.embedding = EmbeddingPosicional(d_model, vocab_size, seq_len)
        self.ff = nn.ModuleList([FeedForward(d_model) for _ in range(nb_layers)])
        self.atts = nn.ModuleList([SelfAttention(d_model, n_head, d_head) for _ in range(nb_layers)])
        self.norm_final = nn.LayerNorm(d_model)
    
    def forward(self, x):
        x = self.embedding(x)
        for ff_layer, att_layer in zip(self.ff, self.atts):
            att_out, _ = att_layer(x)
            x = x + att_out
            ff_out = ff_layer(x)
            x = x + ff_out
        x = self.norm_final(x)
        return x

In [7]:
encoder = Encoder()
x = torch.randint(0, 20, (2, 32))
out = encoder(x)
print(out.shape)

torch.Size([2, 32, 16])


## Tokenizador

In [ ]:
import re
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

In [9]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
import torch

def train_bpe_tokenizer(pages_list, vocab_size=30000):
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[PAD]", "[SEP]", "[UNK]", "[CLS]", "[MASK]"],
        show_progress=True
    )
    tokenizer.train_from_iterator(pages_list, trainer)
    return tokenizer

class BPETokenizerWrapper:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

        self.PAD_TOKEN = tokenizer.get_vocab()["[PAD]"]
        self.MASK_TOKEN = tokenizer.get_vocab()["[MASK]"]
        self.SEP_TOKEN = tokenizer.get_vocab()["[SEP]"]
        self.CLS_TOKEN = tokenizer.get_vocab()["[CLS]"]
        self.UNK_TOKEN = tokenizer.get_vocab()["[UNK]"]
    
    @property
    def vocab_size(self):
        return len(self.tokenizer.get_vocab())

    def encode(self, text, max_len):
        self.tokenizer.enable_truncation(max_length=max_len)

        encoded = self.tokenizer.encode(text)
        ids = encoded.ids
        
        current_len = len(ids)
        if current_len < max_len:
            padding_needed = max_len - current_len
            ids = ids + [self.PAD_TOKEN] * padding_needed
        return torch.tensor(ids, dtype=torch.long)
    
    def decode(self, tokens):
        if isinstance(tokens, torch.Tensor):
            tokens = tokens.tolist()
        return self.tokenizer.decode(tokens, skip_special_tokens=True)

## Dataset

In [10]:
class BookDataset(Dataset):
    def __init__(self, pages, tokenizer, seq_len):
        self.pages = pages
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.pages)

    def __getitem__(self, idx):
        return self.tokenizer.encode(self.pages[idx], self.seq_len)

# Modelo Final

In [ ]:
class EncoderForMaskedLM(nn.Module):
    def __init__(self, encoder, d_model, vocab_size):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: (batch, seq_len)
        embeddings = self.encoder(x) # (batch, seq_len, d_model)
        prediction_scores = self.head(embeddings) # (batch, seq_len, vocab_size)
        return prediction_scores, embeddings

Función para entrenar el modelo

In [ ]:
from tqdm import tqdm 
def train_model(pages, d_model=256, seq_len=20, epochs=100):

    raw_tokenizer = train_bpe_tokenizer(pages)
    tokenizer = BPETokenizerWrapper(raw_tokenizer)
    dataset = BookDataset(pages, tokenizer, seq_len)

    loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)
    
    print(f"Vocab Size: {tokenizer.vocab_size}")

    base_encoder = Encoder(d_model=d_model, vocab_size=tokenizer.vocab_size, 
                           seq_len=seq_len, nb_layers=6, n_head=8, d_head=d_model//8)
    
    model = EncoderForMaskedLM(base_encoder, d_model, tokenizer.vocab_size)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)

    model.train()
    
    for epoch in tqdm(range(epochs)):
        total_loss = 0
        for batch in loader:
            inputs = batch.to(device) 
            labels = batch.clone().to(device) 
            
            rand = torch.rand(inputs.shape, device=device)
            mask_arr = (rand < 0.15) & (inputs != tokenizer.PAD_TOKEN) # máscara solo en tokens no PAD
            
            if not mask_arr.any():
                continue

            labels[~mask_arr] = -100 # Solo calcular pérdida en posiciones enmascaradas
            inputs[mask_arr] = tokenizer.MASK_TOKEN
            
            outputs, _ = model(inputs) 
            
            loss = criterion(outputs.view(-1, tokenizer.vocab_size), labels.view(-1))
            
            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / len(loader)
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    
    torch.save(model.state_dict(), "encoder_mlm.pth")

    return model.encoder, tokenizer

Obtener embeddings de las páginas mediante mean pooling

In [ ]:
import torch.nn.functional as F
def get_page_embeddings(encoder, tokenizer, pages, seq_len):
    encoder.eval()
    embeddings = []
    
    with torch.no_grad():
        for page in pages:
            input_tensor = tokenizer.encode(page, seq_len).unsqueeze(0).to(device)
            
            page_emb_seq = encoder(input_tensor)
            
            mask = (input_tensor != tokenizer.PAD_TOKEN).unsqueeze(-1) # (1, seq_len, 1)
            
            masked_embeddings = page_emb_seq * mask.float()
            
            sum_embeddings = masked_embeddings.sum(dim=1)
            
            token_count = mask.sum(dim=1).clamp(min=1e-9)
            
            page_emb_vector = sum_embeddings / token_count
            mean_embedding = F.normalize(page_emb_vector, p=2, dim=1)
            
            embeddings.append(mean_embedding.squeeze(0).cpu().numpy())            
    return embeddings

In [ ]:
import glob
import pypdf
import re

seq_len = 256
overlap = 10  # Solapamiento de palabras para mantener contexto entre chunks

def clean_page_text(text):
    """
    Limpia el texto crudo extraído del PDF antes de dividirlo.
    """
    if not text: return ""
    
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    text = re.sub(r'[^\w\s.,!?áéíóúÁÉÍÓÚñÑüÜ]', '', text) 

    tokens = word_tokenize(text.lower())
    
    clean_tokens = [t for t in tokens if t.isalnum()]

    text = re.sub(r'\b([b-df-hj-np-tv-zñ])\s+(\w+)', r'\1\2', text, flags=re.IGNORECASE)
    
    text = re.sub(r'(\w+)\s+([b-df-hj-np-tv-zñ])\b', r'\1\2', text, flags=re.IGNORECASE)
    
    return clean_tokens
    

# Cargar páginas
pdf_files = glob.glob("./pdfs/*.pdf")

chunks_pdfs = {}

total_pages = 0

for pdf_file in pdf_files:
    try:
        reader = pypdf.PdfReader(pdf_file)
        file_chunks = []
        for page in tqdm(reader.pages):
            raw_text = page.extract_text(x_tolerance=2, y_tolerance=3)
            # Limpieza inicial
            clean_text = clean_page_text(raw_text)
            
            if not clean_text:
                continue
                
            words = clean_text
            
            # Si la página tiene muy poco contenido se salta
            if len(words) < 10:
                continue
                
            step = seq_len - overlap
            if step < 1: step = 1
            
            for i in range(0, len(words), step):
                # Extraer chunk
                chunk_words = words[i : i + seq_len]
                if len(chunk_words) >= 2 and chunk_words[-2] == "página" and chunk_words[-1].isdigit():
                    chunk_words = chunk_words[:-2]
                # Reconstruir string
                chunk = " ".join(chunk_words)
                
                if len(chunk_words) < 15: # Descartar restos muy pequeños al final
                    continue
                
                file_chunks.append(chunk)
                
        if file_chunks:
            chunks_pdfs[pdf_file] = file_chunks
            print(f"Procesado {pdf_file}: {len(file_chunks)} chunks.")
            
    except Exception as e:
        print(f"Error leyendo {pdf_file}: {e}")

# Verificación
all_chunks = []
for k, v in chunks_pdfs.items():
    all_chunks.extend(v)

print(f"\nTotal chunks limpios: {len(all_chunks)}")
if len(all_chunks) > 0:
    print("Ejemplo de chunk limpio:")
    print(f"'{all_chunks[5]}'")


  0%|          | 0/544 [00:00<?, ?it/s]


  1%|          | 6/544 [00:00<00:10, 49.84it/s]


  3%|▎         | 15/544 [00:00<00:08, 61.04it/s]


  4%|▍         | 22/544 [00:00<00:13, 39.79it/s]


  5%|▍         | 27/544 [00:00<00:17, 30.05it/s]

...


 98%|█████████▊| 627/639 [00:24<00:00, 26.45it/s]


 99%|█████████▊| 631/639 [00:24<00:00, 27.93it/s]


 99%|█████████▉| 635/639 [00:24<00:00, 29.87it/s]


100%|██████████| 639/639 [00:24<00:00, 26.17it/s]

Procesado ./pdfs/El Pozo de la Ascension. (Ed. revisada).pdf: 1216 chunks.

Total chunks limpios: 3438
Ejemplo de chunk limpio:
'son capaces de dominarlos todos ahora kelsier el superviviente el único que ha logrado huir de los pozos de hathsin ha encontrado a vin una pobre chica skaa con mucha suerte tal vez los dos unidos a la rebelión que los skaa intentan desde hace mil años puedan cambiar el mundo y la atroz dominación del lord legislador tras el sorprendente éxito mundial de elantris brandon sanderson ha logrado repetir la hazaña la trilogía nacidos de la bruma mistborn es una excepcional muestra de su nueva fantasía y el primer volumen el imperio final con una conclusión cerrada y grandiosa que permite su lectura como una novela única e independiente nos introduce gratamente en el mundo de los poderes alománticos de la bruma y la ceniza de la dominación y la rebelión con personajes inolvidables como el atormentado keisler en su papel de pigmalión de la más brillante galatea que 

Funciones para obtener todos los chunks, la función para realizar la comparación entre embeddings y para obtener los chunks más importantes

In [15]:
def get_all_chunks(chunks_dict):
    all_chunks = []
    for chunks in chunks_dict.values():
        all_chunks.extend(chunks)
    return all_chunks

def cosine_similarity(v1, v2):
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0: return 0.0
    return np.dot(v1, v2) / (norm_v1 * norm_v2)

def get_relevant_chunks(query, encoder, tokenizer, chunks, seq_len, top_k=5):
    query_emb = get_page_embeddings(encoder, tokenizer, [query], seq_len=seq_len)[0]
    
    similarities = []
    for i, chunk in enumerate(chunks):
        chunk_emb = get_page_embeddings(encoder, tokenizer, [chunk], seq_len=seq_len)[0]
        sim = cosine_similarity(query_emb, chunk_emb)
        similarities.append((sim, chunk))
    
    similarities.sort(reverse=True, key=lambda x: x[0])
    return similarities[:top_k]

## Entrenar el modelo

In [16]:
d_model = 128

def get_all_chunks(chunks_dict):
    all_chunks = []
    for chunks in chunks_dict.values():
        all_chunks.extend(chunks)
    return all_chunks

all_chunks_pdfs = get_all_chunks(chunks_pdfs)


# Entrenar
trained_encoder, tokenizer = train_model(all_chunks_pdfs, d_model=d_model, seq_len=seq_len, epochs=100)

question = "la guerra contra los skaa"
print(f"\nPregunta: {question}")
print(tokenizer.encode(question, seq_len))

relevant_chunks = get_relevant_chunks(question, trained_encoder, tokenizer, all_chunks_pdfs, seq_len=seq_len, top_k=3)

embedding_question = get_page_embeddings(trained_encoder, tokenizer, [question], seq_len=seq_len)[0]

for score, chunk in relevant_chunks:
    print(f"Similitud: {score:.4f}, Chunk: {chunk}")


Vocab Size: 30000



  0%|          | 0/100 [00:00<?, ?it/s]


  1%|          | 1/100 [00:05<08:15,  5.00s/it]


  2%|▏         | 2/100 [00:09<07:50,  4.80s/it]


  3%|▎         | 3/100 [00:14<07:41,  4.75s/it]

...


 99%|█████████▉| 99/100 [07:40<00:04,  4.63s/it]


100%|██████████| 100/100 [07:44<00:00,  4.63s/it]


100%|██████████| 100/100 [07:44<00:00,  4.65s/it]

Epoch 100, Loss: 5.9785

Pregunta: la guerra contra los skaa
tensor([  53, 1904,  340,   79,  432,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    

Similitud: 0.7903, Chunk: parecidas como parte del plan de kelsier para derrocar el imperio final entonces era una criatura nerviosa e insegura preocupada porque aquel mundo recién descubierto de una banda en la que
Similitud: 0.7213, Chunk: por tu propio bien tienes un extraño efecto sobre ellos muchachos como esos deberían estar ansiosos por robar en las aldeas incluso en las pobres sobre todo considerando lo nerviosos que están nuestros hombres y las muchas peleas que ha habido en el campamento y sin embargo no lo hicieron demonios uno de los grupos sintió tanta lástima por los aldeanos que se quedó unos días y les ayudó a regar los campos y a reparar algunas de las casas cett suspiró sacudiendo la cabeza hace unos años me habría reído de quien eligiera la lealtad como base de gobierno pero bueno con el mundo haciéndose pedazos como lo está creo que incluso yo prefiero tener a alguien en quien confiar y no a quien temer supongo que por eso los soldados actúan como lo hacen elend asi

In [17]:
question = "el secreto del lord legislador"
print(f"\nPregunta: {question}")
print(tokenizer.encode(question, seq_len))

relevant_chunks = get_relevant_chunks(question, trained_encoder, tokenizer, all_chunks_pdfs, seq_len=seq_len, top_k=3)

embedding_question = get_page_embeddings(trained_encoder, tokenizer, [question], seq_len=seq_len)[0]

for score, chunk in relevant_chunks:
    print(f"Similitud: {score:.4f}, Chunk: {chunk}")


Pregunta: el secreto del lord legislador
tensor([  58, 1940,  138,  239,  313,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0

Similitud: 0.7779, Chunk: el asedio de luthadel colapso el término usado para referirse a la muerte del lord legislador y la caída del imperio final convento de seran fortaleza de los inquisidores donde sazed y marsh descubrieron las últimas palabras de kwaan cuarto apodo de la moneda de oro imperial procede de la imagen de kedrik shaw el palacio del lord legislador que muestra en su reverso o el cuarto donde vive cubil de la confianza el lugar más sagrado de la tierra natal kandra decantar feruquimia extraer poder de las mentes de metal de un feruquimista es paralelo al término quemar usado por los alománticos demoux oficial del ejército de elend conocido por su fe en el superviviente dockson antigua mano derecha de kelsier miembro de la banda original murió durante el asedio de luthadel
Similitud: 0.7765, Chunk: moneda de oro imperial así llamada por la imagen de kredik shaw el palacio del lord legislador que lleva en su reverso o el cuarto donde vive decantar ferruquimia extraer pod

## Representación de embeddings

In [18]:
from nltk.corpus import stopwords
from collections import Counter


try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

def get_key_words(chunks, top_n=30):
    stop_words = set(stopwords.words('spanish'))
    
    stop_words.update(["si", "tan", "hacia", "entonces", "ahora", "aunque", "después", "toda", "había", "dijo", "volvió", "preguntó", "hizo", "luego", "parecía", "sido", "podía", "demasiado", "menos"])

    all_tokens = []
    
    for chunk in chunks:
        tokens = nltk.word_tokenize(chunk.lower())
        
        filtered = [w for w in tokens if w.isalnum() and w not in stop_words and len(w) > 3]
        all_tokens.extend(filtered)
        
    contador = Counter(all_tokens)
    
    most_common = contador.most_common(top_n)
    palabras_top = [w for w, count in most_common]
    
    return palabras_top

key_words = get_key_words(all_chunks_pdfs, top_n=30)

In [19]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA

categories = {
    'Characters': ['vin', 'kelsier', 'elend', 'sazed', 'dockson', 'ham', 'breeze', 'marsh', 'lord', 'legislador', 'rashek', 'cett', 'straff', 'yeden', 'fantasma'],
    'Entities': ['skaa', 'noble', 'nobles', 'inquisidor', 'koloss', 'kandra', 'terris', 'brumoso', 'nacido', 'soldados', 'hombres', 'gente'],
    'Objects': ['alomancia', 'metal', 'bruma', 'ceniza', 'atium', 'peltre', 'hierro', 'acero', 'moneda'],
    'Places': ['luthadel', 'imperio', 'pozo', 'ascensión', 'ciudad', 'palacio', 'casa'],
    'Verbs': ['matar', 'luchar', 'morir', 'atacar', 'huir', 'destruir', 'hacer', 'ver', 'miró', 'preguntó', 'asintió', 'pensó', 'hizo'],
    'Others': [] 
}

def get_label_and_color(word):
    for cat, lista in categories.items():
        if word in lista:
            return cat
    return 'Others'

colors_map = {
    'Characters': 'red',
    'Entities': 'orange',
    'Objects': 'blue',
    'Places': 'green',
    'Verbs': 'purple',
    'Others': 'gray'
}


def exportar_datos_streamlit(encoder, tokenizer, key_words, file_name="embeddings_data.csv"):
    vectors = []
    labels = []
    cats = []
    
    
    embedding_matrix = encoder.embedding.embedding.weight.data.cpu().numpy()
    
    
    valid_words = []
    for word in key_words:
        try:
            token_ids = tokenizer.encode(word, max_len=5).tolist()
            token_id = token_ids[0]
            if token_id == tokenizer.UNK_TOKEN: continue
            
            vec = embedding_matrix[token_id]
            categorie = get_label_and_color(word)

            vectors.append(vec)
            labels.append(word)
            cats.append(categorie)
        except Exception as e:
            continue
    
    if not vectors:
        return

    pca = PCA(n_components=2)
    result = pca.fit_transform(np.array(vectors))
    
    df = pd.DataFrame({
        'word': labels,
        'x': result[:, 0],
        'y': result[:, 1],
        'category': cats
    })
    
    df.to_csv(file_name, index=False)

exportar_datos_streamlit(trained_encoder, tokenizer, key_words, "trained_data.csv")
encoder_inicial = Encoder(
    d_model=128,                    
    vocab_size=tokenizer.vocab_size, 
    seq_len=256,                    
    nb_layers=6, 
    n_head=8, 
    d_head=16                       
)
exportar_datos_streamlit(trained_encoder, tokenizer, key_words, "initial_data.csv")

## Experimentos con pooling

In [20]:
import torch.nn.functional as F
def get_page_embeddings(encoder, tokenizer, pages, seq_len, pooling='mean'):
    encoder.eval()
    embeddings = []
    
    with torch.no_grad():
        for page in pages:
            # Codificar y mover a GPU
            input_tensor = tokenizer.encode(page, seq_len).unsqueeze(0).to(device)
            
            # Obtener salidas del encoder (1, seq_len, d_model)
            page_emb_seq = encoder(input_tensor)
            
            mask = (input_tensor != tokenizer.PAD_TOKEN).unsqueeze(-1) # (1, seq_len, 1)
            
            masked_embeddings = page_emb_seq * mask.float()
            
            sum_embeddings = masked_embeddings.sum(dim=1)
            
            token_count = mask.sum(dim=1).clamp(min=1e-9)
            
            page_emb_vector = sum_embeddings / token_count
            if pooling == 'mean':
                mean_embedding = F.normalize(page_emb_vector, p=2, dim=1)
            elif pooling == 'max':
                max_embedding, _ = torch.max(masked_embeddings, dim=1)
                mean_embedding = F.normalize(max_embedding, p=2, dim=1)
            else:
                raise ValueError("Pooling method not recognized. Use 'mean' or 'max'.")
            
            embeddings.append(mean_embedding.squeeze(0).cpu().numpy())            
    return embeddings

In [ ]:
import random

def evaluar_modelo(encoder, tokenizer, paginas, seq_len, k=5, num_muestras=50, ratio_query=0.5, pooling='mean'):
    """
    ratio_query: 0.5 significa que usamos la mitad del texto como consulta.
                 0.1 significa que usamos el 10% del texto.
    """
    print(f"--- Evaluación (Ratio: {ratio_query}, Top-K: {k}) ---")
    
    lista_numpy = get_page_embeddings(encoder, tokenizer, paginas, seq_len, pooling=pooling)
    db_embeddings = torch.tensor(np.array(lista_numpy))
    
    aciertos = 0
    indices_prueba = list(range(len(paginas)))
    random.shuffle(indices_prueba)
    indices_prueba = indices_prueba[:num_muestras]
    
    for idx_real in indices_prueba:
        texto_chunk = paginas[idx_real]
        palabras = texto_chunk.split()
        
        if len(palabras) < 10: continue

        # Calculamos longitud basada en el porcentaje
        target_len = int(len(palabras) * ratio_query)
        
        len_query = max(target_len, 5)
        len_query = min(len_query, seq_len - 2)
        
        # Si el chunk es mas corto que seq_len, limitamos
        limit_start = len(palabras) - len_query
        if limit_start <= 0: 
            start_idx = 0
            len_query = len(palabras) # Usar todo si es muy corto
        else:
            start_idx = random.randint(0, limit_start)
            
        palabras_query = palabras[start_idx : start_idx + len_query]
        query_text = " ".join(palabras_query)
        
        vec_query_np = get_page_embeddings(encoder, tokenizer, [query_text], seq_len, pooling=pooling)[0]
        vec_query = torch.tensor(vec_query_np).unsqueeze(0)
        
        similitudes = F.cosine_similarity(vec_query, db_embeddings)
        vals, top_indices = torch.topk(similitudes, k=k)
        
        if idx_real in top_indices.tolist():
            aciertos += 1

    accuracy = (aciertos / len(indices_prueba)) * 100
    print(f"Ratio {ratio_query} -> Precisión: {accuracy:.2f}%")
    return accuracy

Experimentos topk con mean pooling y max pooling

In [ ]:
print("EXPERIMENTOS K=1")

print("EXPERIMENTO 1: Query Larga (50% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.5)
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.3)
                    for _ in range(3)]) / 3

print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.1)
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("EXPERIMENTOS K=3")

print("EXPERIMENTO 1: Query Larga (50% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.5)
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.3)
                    for _ in range(3)]) / 3

print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.1)
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")


print("EXPERIMENTOS K=5")
print("EXPERIMENTO 1: Query Larga (50% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.5)
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.3)
                    for _ in range(3)]) / 3

print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.1)
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")


# Experimentos con max pooling

print("EXPERIMENTOS K=1")

print("EXPERIMENTO 1: Query Larga (50% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.5, pooling='max')
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.3, pooling='max')
                    for _ in range(3)]) / 3

print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=1, ratio_query=0.1, pooling='max')
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("EXPERIMENTOS K=3")

print("EXPERIMENTO 1: Query Larga (50% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.5, pooling='max')
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.3, pooling='max')
                    for _ in range(3)]) / 3

print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=3, ratio_query=0.1, pooling='max')
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")


print("EXPERIMENTOS K=5")
print("EXPERIMENTO 1: Query Larga (50% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.5, pooling='max')
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 2: Query Media (30% del texto original)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.3, pooling='max')
                    for _ in range(3)]) / 3

print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

print("\nEXPERIMENTO 3: Query Corta (10% del texto original - Difícil)")
acc_media = np.sum([evaluar_modelo(trained_encoder, tokenizer, all_chunks_pdfs, seq_len, k=5, ratio_query=0.1, pooling='max')
                    for _ in range(3)]) / 3
print(f"Precisión promedio (3 ejecuciones): {acc_media:.2f}%")

EXPERIMENTOS K=1
EXPERIMENTO 1: Query Larga (50% del texto original)
--- Evaluación (Ratio: 0.5, Top-K: 1) ---


Ratio 0.5 -> Precisión: 60.00%
--- Evaluación (Ratio: 0.5, Top-K: 1) ---


Ratio 0.5 -> Precisión: 54.00%
--- Evaluación (Ratio: 0.5, Top-K: 1) ---


Ratio 0.5 -> Precisión: 60.00%
Precisión promedio (3 ejecuciones): 58.00%

EXPERIMENTO 2: Query Media (30% del texto original)
--- Evaluación (Ratio: 0.3, Top-K: 1) ---


Ratio 0.3 -> Precisión: 14.00%
--- Evaluación (Ratio: 0.3, Top-K: 1) ---


...

Ratio 0.3 -> Precisión: 44.00%
--- Evaluación (Ratio: 0.3, Top-K: 5) ---


Ratio 0.3 -> Precisión: 40.00%
Precisión promedio (3 ejecuciones): 40.67%

EXPERIMENTO 3: Query Corta (10% del texto original - Difícil)
--- Evaluación (Ratio: 0.1, Top-K: 5) ---


Ratio 0.1 -> Precisión: 4.00%
--- Evaluación (Ratio: 0.1, Top-K: 5) ---


Ratio 0.1 -> Precisión: 4.00%
--- Evaluación (Ratio: 0.1, Top-K: 5) ---


Ratio 0.1 -> Precisión: 2.00%
Precisión promedio (3 ejecuciones): 3.33%


In [23]:
query = "la guerra contra los skaa"
print(f"Query: '{query}'\n")

print("Calculando resultados con MEAN Pooling...")
vec_query_mean = get_page_embeddings(trained_encoder, tokenizer, [query], seq_len, pooling='mean')[0]
sims_mean = []

chunk_sample = all_chunks_pdfs[:500] 

for chunk in chunk_sample:
    vec_chunk = get_page_embeddings(trained_encoder, tokenizer, [chunk], seq_len, pooling='mean')[0]
    sim = cosine_similarity(vec_query_mean, vec_chunk)
    sims_mean.append((sim, chunk))

# 2. Calcular con MAX Pooling
print("Calculando resultados con MAX Pooling...")
vec_query_max = get_page_embeddings(trained_encoder, tokenizer, [query], seq_len, pooling='max')[0]
sims_max = []

for chunk in chunk_sample:
    vec_chunk = get_page_embeddings(trained_encoder, tokenizer, [chunk], seq_len, pooling='max')[0]
    sim = cosine_similarity(vec_query_max, vec_chunk)
    sims_max.append((sim, chunk))

# 3. Mostrar Top 3 de cada uno
sims_mean.sort(key=lambda x: x[0], reverse=True)
sims_max.sort(key=lambda x: x[0], reverse=True)

print("\n--- RESULTADOS MEAN POOLING ---")
for i, (score, txt) in enumerate(sims_mean[:3]):
    print(f"Top {i+1} (Score: {score:.4f}): {txt[:100]}...")

print("\n--- RESULTADOS MAX POOLING ---")
for i, (score, txt) in enumerate(sims_max[:3]):
    print(f"Top {i+1} (Score: {score:.4f}): {txt[:100]}...")

Query: 'la guerra contra los skaa'

Calculando resultados con MEAN Pooling...


Calculando resultados con MAX Pooling...



--- RESULTADOS MEAN POOLING ---
Top 1 (Score: 0.6841): para mary beth sanderson que lee fantasía desde antes de que yo naciera y merece plenamente tener un...
Top 2 (Score: 0.6671): y un par de cartas de crédito por valor de diez mil cuartos se lo guardó todo en el bolsillo palpó e...
Top 3 (Score: 0.6464): de la novedad y son somos muchos muchos más de lo que suelen pensar una gran mayoría de editores ést...

--- RESULTADOS MAX POOLING ---
Top 1 (Score: 0.8271): para mary beth sanderson que lee fantasía desde antes de que yo naciera y merece plenamente tener un...
Top 2 (Score: 0.8265): aristócrata te habría convertido en una de las personas más letales e influyentes del imperio final ...
Top 3 (Score: 0.8153): y si es demasiado pesado podrías lastimarte seriamente muchos violentos han sofocado una fea herida ...
